# Data Preparation for ML Modeling


## Section 1.1: Environment Setup

Configure the Cloud compute profile with 32 CPUs, 208 GB RAM, 1,000 GB disk, and a standard VM compute type.

Set `WORKSPACE_CDR` before running the SQL cells. It identifies the project and CDR version used by the queries. To use a later CDR release, update the version suffix after confirming that the required table and column names are unchanged.

### Expected runtime
Measured on the recommended VM (32 CPUs, 208 GB RAM) with CDR v8.

| Run | Participants | Approx. time |
|---|---|---|
| TEST = True | 50 | ~2 min |
| One chunk | ~1,000 | ~15 min |
| Full run | 37 EHR chunks (~36,600) | ~8–9 hours |

In [ ]:
%pip install --upgrade google-api-core google-cloud-bigquery pandas-gbq grpcio grpcio-status ujson

In [ ]:
## Install and import all required libraries
import os
import subprocess

os.environ["BIGQUERY_STORAGE_API_ENABLED"] = "true"
# CDR version used by this workspace
os.environ["WORKSPACE_CDR"] = 'wb-silky-artichoke-2408.C2024Q3R9'
# Workspace's own bucket (works in any copy of the workspace)
os.environ["WORKSPACE_BUCKET"] = subprocess.check_output(['wb', 'resource', 'resolve', '--id=multimodal_risk_prediction']).decode().strip()

import glob
import numpy as np
import h5py
import pandas as pd
import json
import re
import time
from google.cloud import storage
from google.cloud import bigquery
from tqdm.notebook import tqdm
from joblib import Parallel, delayed
import multiprocessing
import matplotlib.pyplot as plt
from collections import defaultdict
from datetime import datetime, timedelta
from scipy import stats
import pandas_gbq
import ujson

## Section 1.2: Calculate Daily Summaries

This section uses the HDF5 files written by notebook 002 to create one row per participant-day:

1. Extract and clean EHR weight, height, and BMI measurements.
2. Summarize daily heart rate, steps, sleep levels, and activity measures.
3. Write per-participant JSON files and combine them into one DataFrame.
4. Merge BMI, sex-at-birth encoding, and last EHR encounter dates, then save the ML input table.

### Section 1.2.1: BMI from the EHR

BMI is derived from EHR measurement records. One query retrieves weight and height measurements, and another retrieves directly recorded BMI values. Weight and height may be recorded in many units (for example g, kg, lb, oz, stone, cm, m, in, or ft, including UCUM forms such as `[lb_av]` and `[in_i]`), so values are converted to kg and cm; rows with units that cannot be converted are dropped and reported. Each weight is then paired with the nearest height within 5 years to calculate BMI, and the BMI is dated at the weight measurement. Directly recorded BMI values are kept even when their unit is missing or unrecognized. The two sources are then pooled, outliers are removed per person using the 1.5 IQR rule, and values outside 10-80 are excluded.

In [ ]:
# Define helper functions for BMI extraction: outlier removal (IQR method) and weight-height to BMI conversion
def _remove_outliers_iqr_per_person(df, value_col, id_col="person_id", k=1.5, min_n=4):
    df = df.copy()

    def _filter(g):
        # Skip filtering when there are too few values
        if g[value_col].notna().sum() < min_n:
            return g

        q1 = g[value_col].quantile(0.25)
        q3 = g[value_col].quantile(0.75)
        iqr = q3 - q1

        # Keep all values when the IQR is 0 (all values identical)
        if iqr == 0 or pd.isna(iqr):
            return g

        low = q1 - k * iqr
        high = q3 + k * iqr
        return g[(g[value_col] >= low) & (g[value_col] <= high)]

    return df.groupby(id_col, group_keys=False).apply(_filter)

# --- BMI unit tables ---
_WEIGHT_TO_KG = {
    'kg': 1.0, 'kilogram': 1.0, 'kilograms': 1.0, 'kgs': 1.0,
    'g': 1e-3, 'gram': 1e-3, 'grams': 1e-3, 'gm': 1e-3,
    'mg': 1e-6, 'milligram': 1e-6,
    'lb': 0.45359237, 'lbs': 0.45359237, 'pound': 0.45359237, 'pounds': 0.45359237,
    '[lb_av]': 0.45359237, '[lb_us]': 0.45359237, 'lb_av': 0.45359237,
    'oz': 0.028349523, 'ounce': 0.028349523, 'ounces': 0.028349523, '[oz_av]': 0.028349523,
    'st': 6.35029318, 'stone': 6.35029318,
}
_HEIGHT_TO_CM = {
    'cm': 1.0, 'centimeter': 1.0, 'centimeters': 1.0,
    'm': 100.0, 'meter': 100.0, 'meters': 100.0, 'metre': 100.0,
    'mm': 0.1, 'millimeter': 0.1, 'millimeters': 0.1,
    'in': 2.54, 'inch': 2.54, 'inches': 2.54, '[in_i]': 2.54, '"': 2.54,
    'ft': 30.48, 'foot': 30.48, 'feet': 30.48, '[ft_i]': 30.48, "'": 30.48,
}


def _norm_unit(u):
    """Lowercase/strip a unit string for table lookup; NaN/None -> '' (missing)."""
    if u is None or (isinstance(u, float) and pd.isna(u)):
        return ''
    return str(u).strip().lower()


def _merge_bmi_weight_height(wh_df, bmi_df, min_bmi_val=10, max_bmi_val=80,
                             pair_tolerance_days=1825, bmi_unit_conversions=None):
    """Merge BMI with weight and height data:
    - converts weight/height using full unit tables (unconvertible units are
      reported and dropped);
    - pairs each weight with the nearest height within ``pair_tolerance_days``
      (default 5 years) per person, and dates the BMI at the WEIGHT date;
    - adds direct BMI measurements, kept even when their unit is missing/unknown;
    - removes per-person IQR outliers and keeps BMI in [min_bmi_val, max_bmi_val].
    """
    wh_df = wh_df.copy()
    wh_df['unit_norm'] = wh_df['unit_source_value'].map(_norm_unit)

    # Weight vs height by standard concept_id, with concept-name fallback.
    cid = wh_df['measurement_concept_id'] if 'measurement_concept_id' in wh_df.columns else pd.Series(index=wh_df.index, dtype='float')
    is_wt = (cid == 3025315) | (wh_df['standard_concept_name'] == 'Body weight')
    is_ht = (cid == 3036277) | (wh_df['standard_concept_name'] == 'Body height')

    wh_df['value_in_kg_cm'] = np.nan
    wh_df.loc[is_wt, 'value_in_kg_cm'] = (
        wh_df.loc[is_wt, 'value_as_number'] * wh_df.loc[is_wt, 'unit_norm'].map(_WEIGHT_TO_KG))
    wh_df.loc[is_ht, 'value_in_kg_cm'] = (
        wh_df.loc[is_ht, 'value_as_number'] * wh_df.loc[is_ht, 'unit_norm'].map(_HEIGHT_TO_CM))

    unconv_wt = wh_df.loc[is_wt & wh_df['value_in_kg_cm'].isna(), 'unit_norm'].value_counts()
    unconv_ht = wh_df.loc[is_ht & wh_df['value_in_kg_cm'].isna(), 'unit_norm'].value_counts()
    if len(unconv_wt):
        print('WARNING: unconvertible WEIGHT units (rows dropped) ->', unconv_wt.to_dict())
    if len(unconv_ht):
        print('WARNING: unconvertible HEIGHT units (rows dropped) ->', unconv_ht.to_dict())

    wh_df['date'] = pd.to_datetime(wh_df['measurement_date']).dt.tz_localize(None)
    wt_df = (wh_df.loc[is_wt & wh_df['value_in_kg_cm'].notna()]
             .rename(columns={'date': 'wt_date'}).sort_values('wt_date', kind='mergesort').reset_index(drop=True))
    ht_df = (wh_df.loc[is_ht & wh_df['value_in_kg_cm'].notna()]
             .rename(columns={'date': 'ht_date'}).sort_values('ht_date', kind='mergesort').reset_index(drop=True))

    print(f'Pairing each weight with the nearest height within {pair_tolerance_days} days (per person)')
    paired = pd.merge_asof(
        wt_df[['person_id', 'wt_date', 'value_in_kg_cm']],
        ht_df[['person_id', 'ht_date', 'value_in_kg_cm']],
        left_on='wt_date', right_on='ht_date', by='person_id',
        direction='nearest', tolerance=pd.Timedelta(days=pair_tolerance_days),
        suffixes=('_wt', '_ht'))

    paired['bmi'] = paired['value_in_kg_cm_wt'] / ((paired['value_in_kg_cm_ht'] / 100) ** 2)
    paired = paired.loc[np.isfinite(paired['bmi'])]
    paired['date'] = paired['wt_date']   # BMI dated at the weight measurement

    # Direct BMI measurements: keep even if the unit is missing/unknown.
    bmi_df = bmi_df.copy()
    bmi_df['date'] = pd.to_datetime(bmi_df['measurement_date']).dt.tz_localize(None)
    bmi_df['bmi'] = bmi_df['value_as_number']
    if 'unit_source_value' in bmi_df.columns:
        bmi_df['unit_norm'] = bmi_df['unit_source_value'].map(_norm_unit)
        print('Direct-BMI unit distribution:', bmi_df['unit_norm'].value_counts(dropna=False).to_dict())
        conv = {k.strip().lower(): v for k, v in (bmi_unit_conversions or {}).items()}
        if conv:
            mult = bmi_df['unit_norm'].map(conv)
            bmi_df.loc[mult.notna(), 'bmi'] = bmi_df.loc[mult.notna(), 'bmi'] * mult[mult.notna()]

    bmi_all_df = pd.concat([bmi_df[['person_id', 'date', 'bmi']],
                            paired[['person_id', 'date', 'bmi']]], ignore_index=True)

    print('Filtering BMI outliers per-person (IQR)')
    bmi_all_df = _remove_outliers_iqr_per_person(bmi_all_df, value_col='bmi')
    print(f'Keeping BMI in [{min_bmi_val}, {max_bmi_val}]')
    bmi_all_df = bmi_all_df.loc[(bmi_all_df['bmi'] >= min_bmi_val) & (bmi_all_df['bmi'] <= max_bmi_val)]
    return bmi_all_df

In [ ]:
# Cohort definitions: participants with Fitbit heart-rate data and EHR data (core CDR tables)
CDR = os.environ["WORKSPACE_CDR"]

# people with any Fitbit data
FITBIT_PERSONS_SQL = " UNION DISTINCT ".join(
    f"SELECT person_id FROM `{CDR}.{t}`"
    for t in ["activity_summary", "heart_rate_summary", "heart_rate_minute_level",
              "steps_intraday", "sleep_daily_summary", "sleep_level"])

# people with any EHR record (site-submitted or participant-mediated)
EHR_PERSONS_SQL = " UNION DISTINCT ".join(
    f"""SELECT t.person_id FROM `{CDR}.{t}` t
        JOIN `{CDR}.{t}_ext` e ON t.{t}_id = e.{t}_id
        WHERE (LOWER(e.src_id) LIKE 'ehr site%' OR LOWER(e.src_id) LIKE 'participant mediated ehr%')"""
    for t in ["condition_occurrence", "measurement", "drug_exposure", "procedure_occurrence",
              "visit_occurrence", "observation", "device_exposure"])

# people with Fitbit heart-rate data and EHR data, used by both BMI queries
BMI_PERSONS_SQL = f"""
    SELECT DISTINCT person_id FROM `{CDR}.heart_rate_minute_level`
    WHERE person_id IN ({FITBIT_PERSONS_SQL})
      AND person_id IN ({EHR_PERSONS_SQL})"""

In [ ]:
%%time
# Extract weight and height measurements from EHR 

# This query represents dataset "Fitbit EHR weight height" for domain "measurement" and was generated for All of Us Controlled Tier Dataset v8
dataset_27640508_measurement_sql = """
    SELECT
        measurement.person_id,
        measurement.measurement_concept_id,
        m_standard_concept.concept_name as standard_concept_name,
        m_standard_concept.concept_code as standard_concept_code,
        m_standard_concept.vocabulary_id as standard_vocabulary,
        measurement.measurement_date,
        measurement.measurement_type_concept_id,
        m_type.concept_name as measurement_type_concept_name,
        measurement.operator_concept_id,
        m_operator.concept_name as operator_concept_name,
        measurement.value_as_number,
        measurement.value_as_concept_id,
        m_value.concept_name as value_as_concept_name,
        measurement.unit_concept_id,
        m_unit.concept_name as unit_concept_name,
        measurement.range_low,
        measurement.range_high,
        measurement.visit_occurrence_id,
        m_visit.concept_name as visit_occurrence_concept_name,
        measurement.measurement_source_value,
        measurement.measurement_source_concept_id,
        m_source_concept.concept_name as source_concept_name,
        m_source_concept.concept_code as source_concept_code,
        m_source_concept.vocabulary_id as source_vocabulary,
        measurement.unit_source_value,
        measurement.value_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (SELECT descendant_concept_id
                    FROM `""" + os.environ["WORKSPACE_CDR"] + """.concept_ancestor`
                    WHERE ancestor_concept_id IN (3025315, 3036277))
            )  
            AND (
                measurement.person_id IN (""" + BMI_PERSONS_SQL + """)
            )) measurement 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_standard_concept 
            ON measurement.measurement_concept_id = m_standard_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_type 
            ON measurement.measurement_type_concept_id = m_type.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_operator 
            ON measurement.operator_concept_id = m_operator.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_value 
            ON measurement.value_as_concept_id = m_value.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_unit 
            ON measurement.unit_concept_id = m_unit.concept_id 
    LEFT JOIn
        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
            ON measurement.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_visit 
            ON v.visit_concept_id = m_visit.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_source_concept 
            ON measurement.measurement_source_concept_id = m_source_concept.concept_id"""

dataset_27640508_measurement_df = pandas_gbq.read_gbq(
    dataset_27640508_measurement_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_27640508_measurement_df.head(5)

In [ ]:
%%time
# Extract direct BMI measurements from EHR 

# This query represents dataset "Fitbit BMI" for domain "measurement" and was generated for All of Us Controlled Tier Dataset v8
dataset_77080212_measurement_sql = """
    SELECT
        measurement.person_id,
        measurement.measurement_concept_id,
        m_standard_concept.concept_name as standard_concept_name,
        m_standard_concept.concept_code as standard_concept_code,
        m_standard_concept.vocabulary_id as standard_vocabulary,
        measurement.measurement_date,
        measurement.measurement_type_concept_id,
        m_type.concept_name as measurement_type_concept_name,
        measurement.operator_concept_id,
        m_operator.concept_name as operator_concept_name,
        measurement.value_as_number,
        measurement.value_as_concept_id,
        m_value.concept_name as value_as_concept_name,
        measurement.unit_concept_id,
        m_unit.concept_name as unit_concept_name,
        measurement.range_low,
        measurement.range_high,
        measurement.visit_occurrence_id,
        m_visit.concept_name as visit_occurrence_concept_name,
        measurement.measurement_source_value,
        measurement.measurement_source_concept_id,
        m_source_concept.concept_name as source_concept_name,
        m_source_concept.concept_code as source_concept_code,
        m_source_concept.vocabulary_id as source_vocabulary,
        measurement.unit_source_value,
        measurement.value_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (SELECT descendant_concept_id
                    FROM `""" + os.environ["WORKSPACE_CDR"] + """.concept_ancestor`
                    WHERE ancestor_concept_id IN (3038553))
            )  
            AND (
                measurement.person_id IN (""" + BMI_PERSONS_SQL + """)
            )) measurement 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_standard_concept 
            ON measurement.measurement_concept_id = m_standard_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_type 
            ON measurement.measurement_type_concept_id = m_type.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_operator 
            ON measurement.operator_concept_id = m_operator.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_value 
            ON measurement.value_as_concept_id = m_value.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_unit 
            ON measurement.unit_concept_id = m_unit.concept_id 
    LEFT JOIn
        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
            ON measurement.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_visit 
            ON v.visit_concept_id = m_visit.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_source_concept 
            ON measurement.measurement_source_concept_id = m_source_concept.concept_id"""

dataset_77080212_measurement_df = pandas_gbq.read_gbq(
    dataset_77080212_measurement_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_77080212_measurement_df.head(5)

In [ ]:
%%time
# Process BMI: convert units, merge weight-height pairs, calculate BMI, remove outliers, filter range 10-80
wh_df = dataset_27640508_measurement_df
bmi_df = dataset_77080212_measurement_df

fitbit_bmi_filt_clean = _merge_bmi_weight_height(wh_df, bmi_df, min_bmi_val=10, max_bmi_val=80)
fitbit_bmi_filt_clean = fitbit_bmi_filt_clean.rename(columns={"date": "bmi_date"})
print(f"Clean BMI records: {len(fitbit_bmi_filt_clean)}")
fitbit_bmi_filt_clean['bmi'].hist(bins=50)

### Section 1.2.2: Sleep Levels

The `sleep_array` in each hd5 file holds one code per minute of the day. Notebook 002 writes codes 0 to 7 for the main sleep labels (deep, wake, light, rem, asleep, restless, awake, unknown), codes 8 to 15 for the same labels outside the main sleep, NaN for a minute with no sleep record, and 88 for an unrecognized label.

The next cell defines that mapping and sets `USE_MAIN_SLEEP_ONLY`, which controls whether summary measures use main sleep only or both main and non-main sleep. Per-level minutes are always saved separately: `deep` is main sleep and `nonmain_deep` is non-main sleep. `deep_total` (and the other `<level>_total` columns) follows `USE_MAIN_SLEEP_ONLY`: main sleep only when `True`, main plus non-main when `False`.

`awake_minutes` is time spent in short awakenings inside a sleep record, represented by the `wake` and `awake` labels.

In [ ]:
# Sleep-level codes in 'sleep_array' (one code per minute). Must match the mapping in 002.
SLEEP_STAGE_MAPPING = {
    "none": np.nan,
    "deep": 0,
    "wake": 1,
    "light": 2,
    "rem": 3,
    "asleep": 4,
    "restless": 5,
    "awake": 6,
    "unknown": 7,
    # non-main variants
    "nonmain_deep": 8,
    "nonmain_wake": 9,
    "nonmain_light": 10,
    "nonmain_rem": 11,
    "nonmain_asleep": 12,
    "nonmain_restless": 13,
    "nonmain_awake": 14,
    "nonmain_unknown": 15,
}

# NaN = no sleep record for that minute ('none' in 002)
# 88  = code used by 002 for labels not in its mapping; counted as unmapped minutes below
SLEEP_NO_RECORD_CODE = 88

# code -> (level, is_main_sleep)
SLEEP_CODE_MAP = {
    int(code): (name.replace("nonmain_", ""), not name.startswith("nonmain_"))
    for name, code in SLEEP_STAGE_MAPPING.items()
    if name != "none"
}

# The 8 Fitbit sleep levels, in the same order as 002
SLEEP_LEVELS = list(dict.fromkeys(
    name.replace("nonmain_", "") for name in SLEEP_STAGE_MAPPING if name != "none"
))
assert len(SLEEP_LEVELS) == 8 and len(SLEEP_CODE_MAP) == 16, (SLEEP_LEVELS, SLEEP_CODE_MAP)

# Levels counted as asleep (restless included, hence asleep_grpd_rstls)
ASLEEP_LEVELS = {"light", "deep", "rem", "asleep", "restless"}

# Short awakenings inside a sleep record, used for awake_minutes
AWAKENING_LEVELS = {"wake", "awake"}

# Which labels feed sleep_duration, awake_minutes, asleep_grpd_rstls, sleep_start_time
# and the sleep/wake HR split:
#   True  -> main sleep only (codes 0-7)
#   False -> main and non-main sleep (codes 0-15)
# Per-level minutes are saved either way.
USE_MAIN_SLEEP_ONLY = False

_MAIN_CODES = np.array([c for c, (lvl, is_main) in SLEEP_CODE_MAP.items() if is_main], dtype=float)
_NONMAIN_CODES = np.array([c for c, (lvl, is_main) in SLEEP_CODE_MAP.items() if not is_main], dtype=float)


def sleep_level_minutes(sleep_raw):
    """Minutes spent in each sleep level on one day.

    Parameters
    ----------
    sleep_raw : np.ndarray
        Per-minute sleep codes for one day, as written by 002.

    Returns
    -------
    (dict, float)
        Minutes per level ('<level>' = main, 'nonmain_<level>' = non-main), and the
        number of minutes with a code not in SLEEP_CODE_MAP (e.g. 88). NaN minutes
        are skipped.
    """
    minutes = {lvl: 0.0 for lvl in SLEEP_LEVELS}
    minutes.update({f"nonmain_{lvl}": 0.0 for lvl in SLEEP_LEVELS})
    unmapped = 0.0

    if sleep_raw.size:
        codes, counts = np.unique(sleep_raw[~np.isnan(sleep_raw)], return_counts=True)
        for code, n in zip(codes, counts):
            entry = SLEEP_CODE_MAP.get(int(code))
            if entry is None:
                unmapped += float(n)
                continue
            level, is_main = entry
            minutes[level if is_main else f"nonmain_{level}"] += float(n)

    return minutes, unmapped


def sum_levels(minutes, levels, include_nonmain):
    """Total minutes over a set of levels, optionally including the non-main labels."""
    total = sum(minutes[lvl] for lvl in levels)
    if include_nonmain:
        total += sum(minutes[f"nonmain_{lvl}"] for lvl in levels)
    return float(total)


# Self-check: 30 min deep, 10 min wake, 20 min non-main light, 5 min NaN, 3 min code 88
_test = np.array([0] * 30 + [1] * 10 + [10] * 20 + [np.nan] * 5 + [SLEEP_NO_RECORD_CODE] * 3, dtype=float)
_minutes, _unmapped = sleep_level_minutes(_test)
assert _unmapped == 3
assert _minutes["deep"] == 30 and _minutes["wake"] == 10 and _minutes["nonmain_light"] == 20
assert _minutes["light"] == 0 and _minutes["nonmain_deep"] == 0
assert sum_levels(_minutes, AWAKENING_LEVELS, include_nonmain=False) == 10   # main only
assert sum_levels(_minutes, ASLEEP_LEVELS, include_nonmain=True) == 50       # deep (main) + light (non-main)
assert sum_levels(_minutes, ASLEEP_LEVELS, include_nonmain=False) == 30
print(f"sleep helper self-check passed, levels: {SLEEP_LEVELS}")

### Section 1.2.3: Daily Feature Extraction

`process_rolling_new` processes each date in one participant's hd5 file and returns a dictionary of daily measures. It divides the minute arrays into sleep and non-sleep periods, summarizes heart rate and steps, counts minutes in each sleep level, and classifies awake minutes using heart-rate reserve and step count. Awake minutes are those in neither main nor non-main sleep, whatever `USE_MAIN_SLEEP_ONLY` is, so naps never count as awake time. Sleep heart rate follows `USE_MAIN_SLEEP_ONLY`. Age is calculated from the date of birth in the stratification file and determines the reference maximum heart rate for that day.

Days with no heart-rate or step data are skipped. Days where the sleep and heart-rate arrays have different lengths are also skipped and flagged as `len_mismatch`.

In [ ]:
# Per-day feature extraction for one person's hd5 file
def process_rolling_new(qc_file, dob, vl, use_main_sleep_only=USE_MAIN_SLEEP_ONLY):

    person_id = os.path.basename(qc_file).replace(".hd5", "")

    # Non-main sleep only counts toward the summaries when use_main_sleep_only is False
    include_nonmain = not use_main_sleep_only
    scope_codes = _MAIN_CODES if use_main_sleep_only else np.concatenate([_MAIN_CODES, _NONMAIN_CODES])
    all_sleep_codes = np.concatenate([_MAIN_CODES, _NONMAIN_CODES])   # main + non-main codes, used for awake time

    if os.path.exists(qc_file):

        with h5py.File(qc_file, 'r') as a:

            for jjk in range(len(sorted(a.keys()))):

                missing_sleep_flag = False
                len_mismatch = False
                date_str = sorted(a.keys())[jjk]
                ref_date = datetime.strptime(date_str[0:10], "%Y-%m-%d")
                age = ref_date.year - dob.year - ((ref_date.month, ref_date.day) < (dob.month, dob.day))
                max_hr_ref = 220 - age

                date_obj = pd.to_datetime(date_str)
                day_of_week = date_obj.day_name()
                day_of_month = date_obj.day
                month = date_obj.month
                year = date_obj.year

                hr_array = a[date_str]['hr_array_qc'][()]
                steps_array = a[date_str]['steps_array_qc'][()]
                sleep_raw = a[date_str]['sleep_array'][()].astype(float)

                if hr_array.size == 0 or steps_array.size == 0:
                    continue

                if hr_array.size > 0 and steps_array.size > 0:
                    step_nw = steps_array.copy().astype(float)
                    step_nw[np.isnan(hr_array)] = np.nan

                    level_minutes, unmapped_sleep_minutes = sleep_level_minutes(sleep_raw)

                    if sleep_raw.size > 0:
                        # 1 = asleep (in scope), 0 = awake or no record
                        sleep_array = np.isin(sleep_raw, scope_codes).astype(float)
                        # 1 = awake = neither main nor non-main sleep, whatever USE_MAIN_SLEEP_ONLY is
                        awake_array = (~np.isin(sleep_raw, all_sleep_codes)).astype(float)
                    else:
                        sleep_array = np.full(len(hr_array), 0.0)
                        awake_array = np.full(len(hr_array), 1.0)   # no sleep record = awake

                    if np.sum(sleep_array) == 0:
                        missing_sleep_flag = True

                    if len(sleep_array) == len(hr_array):
                        heart_rate_non_sleep = np.where(awake_array == 1, hr_array, np.nan)
                        heart_rate_sleep     = np.where(sleep_array == 1, hr_array, np.nan)
                        steps_non_sleep      = np.where(awake_array == 1, steps_array, np.nan)
                        steps_sleep          = np.where(sleep_array == 1, steps_array, np.nan)
                        steps_nw_non_sleep   = np.where(awake_array == 1, step_nw, np.nan) 
                        steps_nw_sleep       = np.where(sleep_array == 1, step_nw, np.nan)
                    else:
                        len_mismatch = True
                        continue

                    # Summary heart rate and step measures
                    RHR = stats.mode(hr_array, nan_policy='omit').mode
                    RHR_sleep = stats.mode(heart_rate_sleep, nan_policy='omit').mode
                    RHR_non_sleep = stats.mode(heart_rate_non_sleep, nan_policy='omit').mode

                    mean_sleep_hr = np.nanmean(heart_rate_sleep)
                    mean_non_sleep_hr = np.nanmean(heart_rate_non_sleep)
                    mean_hr = np.nanmean(hr_array)

                    # Standard deviations of sleep and wake heart rate
                    std_sleep_hr = np.nanstd(heart_rate_sleep)
                    std_non_sleep_hr = np.nanstd(heart_rate_non_sleep)
                    std_hr = np.nanstd(hr_array)

                    step_count = np.nansum(steps_array)
                    step_nw_count = np.nansum(step_nw)
                    sleep_duration = np.nansum(sleep_array)

                    # Short awakenings (wake + awake) and asleep minutes (light + deep + rem + asleep + restless)
                    if not missing_sleep_flag:
                        awake_minutes = sum_levels(level_minutes, AWAKENING_LEVELS, include_nonmain)
                        asleep_grpd_rstls_minutes = sum_levels(level_minutes, ASLEEP_LEVELS, include_nonmain)
                        # Per-level totals following USE_MAIN_SLEEP_ONLY (True: main only, False: main + non-main)
                        level_totals = {f"{lvl}_total": sum_levels(level_minutes, [lvl], include_nonmain) for lvl in SLEEP_LEVELS}
                    else:
                        awake_minutes = float('nan')
                        asleep_grpd_rstls_minutes = float('nan')
                        level_totals = {f"{lvl}_total": float('nan') for lvl in SLEEP_LEVELS}

                    # Sleep start = last awake-to-asleep transition of the day
                    # (may be inaccurate when sleep crosses midnight or data is missing late in the day; see the caveat below)
                    if not missing_sleep_flag:
                        transitions = np.diff(sleep_array)
                        sleep_onsets = np.where(transitions == 1)[0] + 1
                        if len(sleep_onsets) > 0:
                            sleep_start_time = int(sleep_onsets[-1])
                        else:
                            sleep_start_time = float('nan')
                    else:
                        sleep_start_time = float('nan')

                    N_hr_non_sleep = np.sum(~np.isnan(heart_rate_non_sleep))
                    N_hr_sleep     = np.sum(~np.isnan(heart_rate_sleep))
                    N_hr           = np.sum(~np.isnan(hr_array))

                    N_steps = np.sum(~np.isnan(steps_array))
                    N_steps_nw = np.sum(~np.isnan(step_nw))

                    # Pre-classification arrays
                    hr_clean = heart_rate_non_sleep.astype(float)
                    st_clean = steps_nw_non_sleep.astype(float)

                    nan_time = np.sum(np.where(np.isnan(hr_clean), 1, 0).astype(float))

                    max_mvg_ref_mvg = (40 * (max_hr_ref - RHR)) / 100
                    max_mvg_ref_sd  = (20 * (max_hr_ref - RHR)) / 100

                    # % awake time in low/medium/high HR buckets
                    valid_awake_hr = ~np.isnan(hr_clean)
                    n_valid_awake_hr = np.sum(valid_awake_hr)

                    if n_valid_awake_hr > 0:
                        low_hr_threshold = max_mvg_ref_sd + RHR
                        high_hr_threshold = max_mvg_ref_mvg + RHR

                        awake_hr_low_count  = np.sum(valid_awake_hr & (hr_clean < low_hr_threshold))
                        awake_hr_med_count  = np.sum(valid_awake_hr & (hr_clean >= low_hr_threshold) & (hr_clean < high_hr_threshold))
                        awake_hr_high_count = np.sum(valid_awake_hr & (hr_clean >= high_hr_threshold))

                        pct_awake_hr_low  = float(awake_hr_low_count / n_valid_awake_hr)
                        pct_awake_hr_med  = float(awake_hr_med_count / n_valid_awake_hr)
                        pct_awake_hr_high = float(awake_hr_high_count / n_valid_awake_hr)
                    else:
                        pct_awake_hr_low  = float('nan')
                        pct_awake_hr_med  = float('nan')
                        pct_awake_hr_high = float('nan')

                    # 1-min activity type markers
                    mvpa_1min       = np.where(np.isnan(hr_clean), np.nan,
                                        ((hr_clean >= (max_mvg_ref_mvg+RHR)) & (st_clean > 0)).astype(float))
                    mvpa_20_1min    = np.where(np.isnan(hr_clean), np.nan,
                                        ((hr_clean >= (max_mvg_ref_mvg+RHR)) & (st_clean > 20)).astype(float))
                    mvpa_100_1min   = np.where(np.isnan(hr_clean), np.nan,
                                        ((hr_clean >= (max_mvg_ref_mvg+RHR)) & (st_clean > 100)).astype(float))

                    lact_1min       = np.where(np.isnan(hr_clean), np.nan,
                        ((hr_clean > (max_mvg_ref_sd+RHR)) & (hr_clean < (max_mvg_ref_mvg+RHR)) & (st_clean > 0)).astype(float))

                    sedentary_1min  = np.where(np.isnan(hr_clean), np.nan,
                                        ((hr_clean <= (max_mvg_ref_sd+RHR)) & (st_clean == 0)).astype(float))

                    HRR_40_s0_1min     = np.where(np.isnan(hr_clean), np.nan,
                                            ((hr_clean >= (max_mvg_ref_mvg+RHR)) & (st_clean == 0)).astype(float))

                    HRR_20_40_s0_1min  = np.where(np.isnan(hr_clean), np.nan,
                                            ((hr_clean > (max_mvg_ref_sd+RHR)) & (hr_clean < (max_mvg_ref_mvg+RHR)) & (st_clean == 0)).astype(float))

                    HRR_20_sn0_1min    = np.where(np.isnan(hr_clean), np.nan,
                                            ((hr_clean <= (max_mvg_ref_sd+RHR)) & (st_clean > 0)).astype(float))

                    # Build metrics dictionary for this day
                    metrics_this_day = {

                        "mvpa": {
                            "standard": {
                                "raw": int(np.nansum(mvpa_1min == 1)),
                            },
                            "20_plus": {
                                "raw": int(np.nansum(mvpa_20_1min == 1)),
                            },
                            "100_plus": {
                                "raw": int(np.nansum(mvpa_100_1min == 1)),
                            },
                        },
                        "lact": {
                            "raw": int(np.nansum(lact_1min == 1)),
                        },
                        "sedentary": {
                            "raw": int(np.nansum(sedentary_1min == 1)),
                        },
                        "HRR": {
                            "40_s0": {
                                "raw": int(np.nansum(HRR_40_s0_1min == 1)),
                            },
                            "20_40_s0": {
                                "raw": int(np.nansum(HRR_20_40_s0_1min == 1)),
                            },
                            "20_sn0": {
                                "raw": int(np.nansum(HRR_20_sn0_1min == 1)),
                            },
                        },
                        "summary": {
                            "RHR": float(RHR),
                            "RHR_sleep": float(RHR_sleep),
                            "RHR_non_sleep": float(RHR_non_sleep),
                            "mean_sleep_hr": float(mean_sleep_hr),
                            "mean_non_sleep_hr": float(mean_non_sleep_hr),
                            "mean_hr": float(mean_hr),
                            "std_sleep_hr": float(std_sleep_hr),
                            "std_non_sleep_hr": float(std_non_sleep_hr),
                            "std_hr": float(std_hr),
                            "step_count": float(step_count),
                            "step_count_nw": float(step_nw_count),
                            "sleep_duration": float(sleep_duration),
                            "awake_minutes": float(awake_minutes),
                            "asleep_grpd_rstls": float(asleep_grpd_rstls_minutes),
                            "sleep_start_time": float(sleep_start_time) if not isinstance(sleep_start_time, float) else sleep_start_time,
                            "pct_awake_hr_low": float(pct_awake_hr_low),
                            "pct_awake_hr_med": float(pct_awake_hr_med),
                            "pct_awake_hr_high": float(pct_awake_hr_high),
                            "N_hr": int(N_hr),
                            "N_hr_sleep": int(N_hr_sleep),
                            "N_hr_non_sleep": int(N_hr_non_sleep),
                            "N_steps": int(N_steps),
                            "N_steps_nw": int(N_steps_nw),
                            "non-wear": float(nan_time),
                            "age": int(age),
                            "unmapped_sleep_minutes": float(unmapped_sleep_minutes),
                            # Minutes per sleep level
                            **{k: float(v) for k, v in level_minutes.items()},
                            **level_totals,
                        },
                        "flags": {
                            "missing_sleep": missing_sleep_flag,
                            "len_mismatch": len_mismatch,
                            "main_sleep_only": bool(use_main_sleep_only),
                            "unmapped_sleep_codes": bool(unmapped_sleep_minutes > 0),
                        },
                        "date_info": {
                            "day_of_month": day_of_month,
                            "month": month,
                            "year": year,
                            "day_of_week": day_of_week,   # weekday name, e.g. Monday
                        }
                    }

                    vl['person_id'].append(person_id)
                    vl['date'].append(date_str[:10])
                    vl['metrics'].append(metrics_this_day)

    return vl

#### Sleep Start Time Caveat

`summary.sleep_start_time` is the final awake-to-asleep transition within a single calendar-day array. It can be misleading when a sleep episode crosses midnight or sleep and heart-rate records are missing near the end of the day; for example, a 1 AM onset may represent a prior night's bedtime. Determining the full sleep episode would require matching onset and end-of-sleep markers across adjacent days. The feature is retained because it is included in the ML table and in the validation section.

In [ ]:
# Worker function for joblib: looks up the participant's date of birth and runs the daily extraction
def process_rolling_new_calculation(qc_file, df_summary_i, use_main_sleep_only=USE_MAIN_SLEEP_ONLY):

    """
    Run process_rolling_new for one participant's hd5 file.

    Parameters:
    -----------
    qc_file : str
        Path to the participant's hd5 file (one group per date with
        'hr_array_qc', 'steps_array_qc' and 'sleep_array').

    df_summary_i : pandas.DataFrame
        Stratification table with date_of_birth

    use_main_sleep_only : bool
        True uses main sleep only for the sleep summaries, False adds non-main sleep.

    """

    person_id = os.path.basename(qc_file).replace(".hd5", "")
    dob_str = df_summary_i[df_summary_i['person_id']==int(person_id)]['date_of_birth'].values[0][0:10]
    dob = datetime.strptime(dob_str, "%Y-%m-%d")

    activity_types = ['date','person_id','metrics']
    all_act_lvl = {key: [] for key in activity_types}

    all_act_lvl = process_rolling_new(qc_file, dob, all_act_lvl, use_main_sleep_only=use_main_sleep_only)

    key_list = ['date','person_id','metrics']
    act_lvl_dict = {key: all_act_lvl[key] for key in key_list}

    return act_lvl_dict, f"jsons_rolling/{person_id}.json"

#### Process Participants in Parallel

Each participant's hd5 file is processed in parallel and saved as its own JSON file as soon as it is done. If the run stops, rerunning the cell skips participants that already have a JSON. Participants that fail, or that produce no days, are listed in `errors_003.csv`. To start over, delete the `jsons_rolling` folder first.

`TEST` is set in 001 and read in the next cell. With `TEST = True` the notebook runs on the first 50 participants. A test uses its own folders (`all_qc_local_test`, `jsons_rolling_test`) and saves `daily_activity_stats_ehr_TEST.pq`, so it never changes a full run. With `TEST = True` the input folders also get the `_test` suffix used by 001, 002, 004 and 005 (for example `ingested_with_ehr_test`, `all_person_stats_test`), so the whole chain can be tested together.

In [ ]:
# Load shared input and output paths
# TEST = True: first 50 participants only, output saved as *_TEST.pq. False = full run.
# TEST is set once in 001 and read here, so every notebook uses the same value (run 001's setup cell first)
TEST_FLAG_FILE = '/home/jupyter/workspace/multimodal_risk_prediction/test_flag.txt'
assert os.path.exists(TEST_FLAG_FILE), "TEST flag not found: run the setup cell in 001 first, it saves TEST for 002-005"
TEST = open(TEST_FLAG_FILE).read().strip() == 'True'
bucket_ = os.environ.get('WORKSPACE_BUCKET')

# Name of the output ML table saved to the bucket (.pq)
file_path_to_save = 'daily_activity_stats_ehr_TEST' if TEST else 'daily_activity_stats_ehr'
print(f"TEST={TEST} -> output {bucket_}/{file_path_to_save}.pq")

# The next cell moves to a local working folder; remember the notebook folder so this cell can be rerun
NOTEBOOK_DIR = globals().get('NOTEBOOK_DIR', os.getcwd())
file_paths_values = pd.read_csv(os.path.join(NOTEBOOK_DIR, 'filepaths_values.csv'))
file_paths_values = file_paths_values[
    file_paths_values['notebook'] == 'trends_calculation'
][['notebook', 'key_', 'value_']]
paths_dict = file_paths_values.set_index('key_')['value_']

# TEST reads the '_test' folders/files written by 001/002 in test mode (same suffix rule in 001-005)
if TEST:
    paths_dict = paths_dict.map(lambda v: v if '.' in str(v) else f'{v}_test')

required_path_keys = {'folder_ingest_ehr', 'person_stats_file'}
missing_path_keys = required_path_keys - set(paths_dict.index)
if missing_path_keys:
    raise KeyError(
        f"filepaths_values.csv is missing required keys for notebook 003: {sorted(missing_path_keys)}"
    )

In [ ]:
%%time
# Copy HDF5 files from the bucket and list them
import shutil

# Work on the VM's local disk. If the notebook folder is a bucket mount, reading hd5 files from it is very slow.
RUN_DIR = os.path.expanduser('~/run_003')
os.makedirs(RUN_DIR, exist_ok=True)
os.chdir(RUN_DIR)
print("Working folder:", os.getcwd())

# TEST uses separate folders, so a test never mixes with or deletes a full run
qc_folder = 'all_qc_local_test' if TEST else 'all_qc_local'
JSON_DIR = 'jsons_rolling_test' if TEST else 'jsons_rolling'

if TEST:
    shutil.rmtree(qc_folder, ignore_errors=True)   # fresh test every time
    shutil.rmtree(JSON_DIR, ignore_errors=True)
    print(f"TEST: cleared {qc_folder}/ and {JSON_DIR}/")
else:
    print(f"FULL: {len(glob.glob(f'{JSON_DIR}/*.json')):,} participants already have a JSON and will be skipped")

# Stratification table, used for date of birth and sex at birth
df_summary_i = pd.read_csv(f'{bucket_}/{paths_dict.get("person_stats_file")}.tsv', sep='\t')

folder_to_copy_from = paths_dict.get("folder_ingest_ehr")
src_qc = f"{bucket_}/{folder_to_copy_from}/chunk_*_qc/*.hd5"  # 002 writes one chunk_{n}_qc folder per chunk
os.makedirs(qc_folder, exist_ok=True)
os.makedirs(JSON_DIR, exist_ok=True)

# Copy only the chunk_*_qc folders (not chunk_*_raw, which use the same file names)
if TEST:
    # First 50 files only
    subprocess.run(f"gcloud storage ls '{src_qc}' | head -50 | gcloud storage cp -I {qc_folder}/",
                   shell=True, check=True)
else:
    # Full copy (approximately 230 GB, 15-30 minutes). -n skips files already copied, so reruns are quick.
    subprocess.run(f"gcloud storage cp -n '{src_qc}' {qc_folder}/", shell=True, check=True)

# Recursive glob since files may be in subdirectories
qc_files = sorted(glob.glob(f"{qc_folder}/**/*.hd5", recursive=True))
print(f"Total .hd5 files found: {len(qc_files)}")

names = [os.path.basename(f) for f in qc_files]
assert len(names) == len(set(names)), "The same participant's file appears twice (copied from two folders?)"

# Every participant needs a date of birth in the stratification table (should print 0)
missing = sorted(set(int(n[:-4]) for n in names) - set(df_summary_i['person_id'].astype(int)))
print(f"Participants without a row in the stratification table: {len(missing)}", missing[:10])


#### Sleep Code Preflight Check

The following is a quick check of the sleep codes in the ingested files. It reads a sample of hd5 files and counts the minutes for each sleep code. It stops if a code is not in the notebook 002 mapping, and prints a warning if no NaN minutes or no non-main sleep codes are found.

In [ ]:
# Check the sleep codes in a sample of files against the 002 mapping before the full run
N_FILES_TO_INSPECT = min(20, len(qc_files))

# Inspect the structure of one file
with h5py.File(qc_files[0], 'r') as a:
    days = sorted(a.keys())
    print(f"{qc_files[0]}: {len(days)} days")
    print("datasets per day :", list(a[days[0]].keys()))
    print("sleep_array dtype:", a[days[0]]['sleep_array'].dtype)

# Minutes per code across the sample
code_minutes = defaultdict(float)
nan_minutes = 0.0
for qc_file in tqdm(qc_files[:N_FILES_TO_INSPECT]):
    with h5py.File(qc_file, 'r') as a:
        for date_str in a.keys():
            sleep_raw = a[date_str]['sleep_array'][()].astype(float)
            if sleep_raw.size == 0:
                continue
            nan_minutes += float(np.sum(np.isnan(sleep_raw)))
            codes, counts = np.unique(sleep_raw[~np.isnan(sleep_raw)], return_counts=True)
            for code, n in zip(codes, counts):
                code_minutes[int(code)] += float(n)

print(f"\ncode -> minutes observed in {N_FILES_TO_INSPECT} files -> mapping")
for code in sorted(code_minutes):
    if code == SLEEP_NO_RECORD_CODE:
        label = "(002 fallback for an unrecognized level, not counted as a level)"
    else:
        label = SLEEP_CODE_MAP.get(code, '*** NOT IN 002 MAPPING ***')
    print(f"  {code:>4}: {code_minutes[code]:>12,.0f} min  ->  {label}")
print(f"   NaN: {nan_minutes:>12,.0f} min  ->  no sleep record for that minute")

unknown = sorted(c for c in code_minutes if c not in SLEEP_CODE_MAP and c != SLEEP_NO_RECORD_CODE)
assert not unknown, (
    f"Sleep codes {unknown} are in the data but not in SLEEP_STAGE_MAPPING. "
    f"Update the mapping to match notebook 002 before running the extraction."
)

# Minutes with no sleep record should be stored as NaN
if nan_minutes == 0:
    print("\nWARNING: no NaN minutes found in sleep_array.")
    print("Minutes with no sleep record are expected to be NaN. Check that 002 saved sleep_array as float.")

nonmain_minutes = sum(m for c, m in code_minutes.items()
                      if c in SLEEP_CODE_MAP and not SLEEP_CODE_MAP[c][1])
if nonmain_minutes == 0:
    print("\nWARNING: no non-main sleep codes (8-15) found in the ingested files.")
    print("All 'nonmain_<level>' columns will be 0 and USE_MAIN_SLEEP_ONLY=False changes nothing.")
else:
    print(f"\nnon-main sleep minutes in sample: {nonmain_minutes:,.0f}")

print(f"USE_MAIN_SLEEP_ONLY = {USE_MAIN_SLEEP_ONLY}")

In [ ]:
%%time
# Process all participants in parallel, write per-person JSONs. Each JSON is written as soon as that person is done:
# - a crash or restart can resume: people with an existing JSON are skipped
# - one person failing (e.g. missing date of birth) is logged and skipped instead of stopping the whole run
import sys, joblib
from tqdm import tqdm as tqdm_text   # plain text bar: updates in place, works without notebook widgets

def make_json_serializable(o):
    if isinstance(o, np.generic):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, dict):
        return {k: make_json_serializable(v) for k, v in o.items()}
    if isinstance(o, list):
        return [make_json_serializable(x) for x in o]
    return o

OUT_DIR = os.path.abspath(JSON_DIR)        # absolute path, so worker processes write to the right folder
os.makedirs(OUT_DIR, exist_ok=True)
qc_files = [os.path.abspath(f) for f in qc_files]  # same for the input files

def process_and_save(qc_file):
    out = os.path.join(OUT_DIR, os.path.basename(qc_file).replace('.hd5', '.json'))
    if os.path.exists(out):
        return ('skipped', qc_file, '')            # already done in an earlier run
    try:
        json_data, _ = process_rolling_new_calculation(qc_file, df_summary_i, USE_MAIN_SLEEP_ONLY)
        if not json_data.get('date'):                                  # no days produced
            return ('error', qc_file, 'no days produced')              # log it, don't save an empty JSON
        tmp = out + '.tmp'
        with open(tmp, 'w') as f:
            f.write(ujson.dumps(make_json_serializable(json_data)))
        os.replace(tmp, out)                        # rename only after a complete write, so no half-written JSONs
        return ('done', qc_file, '')
    except Exception as e:
        return ('error', qc_file, f"{type(e).__name__}: {e}")

# Progress counts people as they FINISH (not as jobs are handed out)
mode = 'generator_unordered' if tuple(int(x) for x in joblib.__version__.split('.')[:2]) >= (1, 4) else 'generator'
st = time.time()
results, counts = [], {'done': 0, 'skipped': 0, 'error': 0}
jobs = Parallel(n_jobs=multiprocessing.cpu_count(), return_as=mode)(delayed(process_and_save)(f) for f in qc_files)
with tqdm_text(total=len(qc_files), desc='people', unit='person', mininterval=10, file=sys.stdout) as bar:
    for r in jobs:
        results.append(r)
        counts[r[0]] += 1
        bar.update(1)
        bar.set_postfix(counts, refresh=False)

print(f"{counts}  total files: {len(qc_files)}  time: {(time.time() - st) / 60:.1f} min")

errors = pd.DataFrame([r for r in results if r[0] == 'error'], columns=['status', 'file', 'error'])
errors.to_csv('errors_003.csv', index=False)        # always written, so an old error list can't linger
if len(errors):
    print(f"{len(errors)} people failed; saved to errors_003.csv. Most common errors:")
    print(errors['error'].str[:80].value_counts().head(5).to_string())

### Section 1.2.4: Combine JSON Files into One Table

The per-participant JSON files store metrics as nested dictionaries. This step flattens them into columns such as `summary.mean_hr` and combines all participant-days into one DataFrame. The Parquet file written here is an intermediate checkpoint; BMI, sex-at-birth encoding, and encounter dates are merged before the final save.

In [ ]:
%%time
# Read per-person JSONs, flatten nested metrics, build single DataFrame, save as parquet
# Builds one small table per person, then stacks them.
# Column names look like 'summary.mean_hr' and 'date_info.day_of_week'.
json_files = sorted(glob.glob(f'{JSON_DIR}/*.json'))
print(f"JSON files to combine: {len(json_files):,}")

parts = []                                              # one small DataFrame per person
for fn in tqdm(json_files):
    with open(fn) as f:
        d = json.load(f)                                # {'person_id': [...], 'date': [...], 'metrics': [...]}
    if not d['date']:                                   # skip empty files
        continue
    df_p = pd.json_normalize(d['metrics'])              # flatten nested metrics -> one row per day, '.'-joined column names
    df_p['person_id'] = [int(x) for x in d['person_id']]
    df_p['date'] = d['date']
    parts.append(df_p)

# Stack all people into one table
daily_activity_stats_ehr = pd.concat(parts, ignore_index=True)
del parts                                               # free the per-person pieces

assert len(daily_activity_stats_ehr), f"No rows built: JSONs are empty. Check {JSON_DIR}/."
print(f"{daily_activity_stats_ehr['person_id'].nunique():,} people, "
      f"{len(daily_activity_stats_ehr):,} person-days, {daily_activity_stats_ehr.shape[1]} columns")

# Intermediate checkpoint (overwritten later by the EHR cell with BMI + EHR columns added)
daily_activity_stats_ehr.to_parquet(f'{bucket_}/{file_path_to_save}.pq')

### Section 1.2.5: Add BMI and Sex-at-Birth Encoding

`merge_asof` assigns the most recent BMI on or before each Fitbit day when it falls within five years. Sex at birth is encoded as `1` for Male, `0` for Female, and `NaN` for other or missing values.

In [ ]:
%%time
# Merge BMI (most recent within 5 years via merge_asof) and is_male into the DataFrame

# Convert date columns to datetime
daily_activity_stats_ehr['date'] = pd.to_datetime(daily_activity_stats_ehr['date'])
fitbit_bmi_filt_clean['bmi_date'] = pd.to_datetime(fitbit_bmi_filt_clean['bmi_date'])

daily_activity_stats_ehr = pd.merge_asof(
    daily_activity_stats_ehr.astype({'person_id': int}).sort_values(["date", "person_id"], kind="mergesort").reset_index(drop=True),
    fitbit_bmi_filt_clean[["person_id", "bmi_date", "bmi"]].astype({'person_id': int}).sort_values(["bmi_date", "person_id"], kind="mergesort").reset_index(drop=True),
    left_on="date",
    right_on="bmi_date",
    by="person_id",
    direction="backward",
    tolerance=pd.Timedelta("1825D")  # Up to five years before the Fitbit day
)

# Values other than Male or Female are set to NaN rather than 0
df_summary_i['is_male'] = df_summary_i['sex_at_birth'].map({'Male': 1, 'Female': 0})
daily_activity_stats_ehr = daily_activity_stats_ehr.merge(
    df_summary_i[['person_id', 'is_male']],
    on='person_id',
    how='left'
)

# Check results
print(f"BMI NaN rate: {daily_activity_stats_ehr['bmi'].isna().mean()*100:.1f}%")
print(f"is_male NaN rate: {daily_activity_stats_ehr['is_male'].isna().mean()*100:.1f}%")
print("\nBMI stats:")
print(daily_activity_stats_ehr['bmi'].describe())
print("\nis_male distribution (1=Male, 0=Female; other labels are NaN):")
print(daily_activity_stats_ehr['is_male'].value_counts(dropna=False))

## Section 1.3: EHR History and Follow-up

This section finds the first and most recent condition, procedure, and laboratory measurement dates for each participant. The first dates are used to check that each participant has EHR history on or before their first Fitbit day, so that prevalent disease is not treated as incident disease. Participants without that history are kept and flagged with `has_ehr_before_or_on_first_fitbit = False`, and listed in `missing_ehr_history_003.csv`; filter them in the downstream notebooks. The last dates mark the end of EHR follow-up. Without code filters, the query considers any record in each domain. Provide `condition_codes`, `procedure_codes`, `measurement_codes`, or the corresponding `*_concept_ids` to restrict the query to a phenotype.

In [ ]:
# First and last condition, procedure, and lab dates per participant

def _clean_int_list(values):
    """
    Return sorted unique integers. None is returned unchanged (no filter), which differs from [] (match nothing).
    """
    if values is None:
        return None

    cleaned = []
    for x in values:
        if pd.notna(x):
            cleaned.append(int(x))

    return sorted(set(cleaned))


def _clean_code_list(values):
    """
    Return sorted unique uppercase strings. None is returned unchanged.
    """
    if values is None:
        return None

    cleaned = []
    for x in values:
        if pd.notna(x):
            s = str(x).strip().upper()
            if s:
                cleaned.append(s)

    return sorted(set(cleaned))


def _validate_dataset_id(dataset_id):
    """
    BigQuery does not accept table names as query parameters, so dataset_id is
    inserted directly into the SQL string. Check that it has the form 'dataset' or 'project.dataset'.
    """
    if not re.fullmatch(r"[A-Za-z0-9_-]+(\.[A-Za-z0-9_]+)?", dataset_id):
        raise ValueError(
            "dataset_id must look like 'dataset' or 'project.dataset'. "
            f"Got: {dataset_id!r}"
        )


def pull_ehr_code_date_bounds(
    person_ids,
    dataset_id=None,
    condition_codes=None,
    procedure_codes=None,
    measurement_codes=None,
    condition_concept_ids=None,
    procedure_concept_ids=None,
    measurement_concept_ids=None,
):
    """
    Return the first and last matching condition, procedure, and measurement dates
    for a subset of participant IDs.

    Parameters
    ----------
    person_ids : iterable
        Participant IDs.

    dataset_id : str, optional
        BigQuery dataset. Defaults to os.environ["WORKSPACE_CDR"].

    condition_codes : iterable, optional
        Condition codes to match. Can include ICD or SNOMED codes.
        The query checks:
        - condition_source_value
        - condition_source_concept_id -> concept.concept_code
        - condition_concept_id -> concept.concept_code

    procedure_codes : iterable, optional
        Procedure codes to match. Can include CPT, HCPCS, ICD procedure,
        SNOMED procedure codes, or source values.

    measurement_codes : iterable, optional
        Lab/measurement codes to match. Often LOINC codes, OMOP concept codes,
        or source values.

    condition_concept_ids : iterable, optional
        OMOP condition concept IDs. The query checks both condition_concept_id
        and condition_source_concept_id.

    procedure_concept_ids : iterable, optional
        OMOP procedure concept IDs. The query checks both procedure_concept_id
        and procedure_source_concept_id.

    measurement_concept_ids : iterable, optional
        OMOP measurement concept IDs. The query checks both measurement_concept_id
        and measurement_source_concept_id.

    Returns
    -------
    pd.DataFrame
        One row per person_id with:
        - first_condition_date
        - first_procedure_date
        - first_lab_measurement_date
        - first_any_code_date
        - last_condition_date
        - last_procedure_date
        - last_lab_measurement_date
        - last_any_code_date

    Notes
    -----
    For each domain:
    - If both codes and concept_ids are None, no code filter is applied for
      that domain, so the function returns the first and last dates for any record.
    - If you pass an empty list, that domain will return no matches.
    """

    if dataset_id is None:
        dataset_id = os.environ["WORKSPACE_CDR"]

    _validate_dataset_id(dataset_id)

    cleaned_person_ids = _clean_int_list(person_ids)
    if not cleaned_person_ids:
        return pd.DataFrame(
            columns=[
                "person_id",
                "first_condition_date",
                "first_procedure_date",
                "first_lab_measurement_date",
                "first_any_code_date",
                "last_condition_date",
                "last_procedure_date",
                "last_lab_measurement_date",
                "last_any_code_date",
            ]
        )

    condition_codes_clean = _clean_code_list(condition_codes)
    procedure_codes_clean = _clean_code_list(procedure_codes)
    measurement_codes_clean = _clean_code_list(measurement_codes)

    condition_ids_clean = _clean_int_list(condition_concept_ids)
    procedure_ids_clean = _clean_int_list(procedure_concept_ids)
    measurement_ids_clean = _clean_int_list(measurement_concept_ids)

    match_all_conditions = (
        condition_codes is None and condition_concept_ids is None
    )
    match_all_procedures = (
        procedure_codes is None and procedure_concept_ids is None
    )
    match_all_measurements = (
        measurement_codes is None and measurement_concept_ids is None
    )

    query = f"""
    WITH input_people AS (
      SELECT person_id
      FROM UNNEST(@person_ids) AS person_id
    ),

    input_condition_codes AS (
      SELECT
        UPPER(TRIM(CAST(code AS STRING))) AS code,
        REPLACE(UPPER(TRIM(CAST(code AS STRING))), '.', '') AS code_nodot
      FROM UNNEST(@condition_codes) AS code
      WHERE code IS NOT NULL
        AND TRIM(CAST(code AS STRING)) != ''
    ),

    input_procedure_codes AS (
      SELECT
        UPPER(TRIM(CAST(code AS STRING))) AS code,
        REPLACE(UPPER(TRIM(CAST(code AS STRING))), '.', '') AS code_nodot
      FROM UNNEST(@procedure_codes) AS code
      WHERE code IS NOT NULL
        AND TRIM(CAST(code AS STRING)) != ''
    ),

    input_measurement_codes AS (
      SELECT
        UPPER(TRIM(CAST(code AS STRING))) AS code,
        REPLACE(UPPER(TRIM(CAST(code AS STRING))), '.', '') AS code_nodot
      FROM UNNEST(@measurement_codes) AS code
      WHERE code IS NOT NULL
        AND TRIM(CAST(code AS STRING)) != ''
    ),

    last_conditions AS (
      SELECT
        co.person_id,
        MIN(co.condition_start_date) AS first_condition_date,
        MAX(co.condition_start_date) AS last_condition_date
      FROM `{dataset_id}.condition_occurrence` co
      JOIN input_people ip
        ON co.person_id = ip.person_id
      LEFT JOIN `{dataset_id}.concept` c_standard
        ON co.condition_concept_id = c_standard.concept_id
      LEFT JOIN `{dataset_id}.concept` c_source
        ON co.condition_source_concept_id = c_source.concept_id
      WHERE co.condition_start_date IS NOT NULL
        AND (
          @match_all_conditions

          OR co.condition_concept_id IN UNNEST(@condition_concept_ids)
          OR co.condition_source_concept_id IN UNNEST(@condition_concept_ids)

          OR UPPER(TRIM(CAST(co.condition_source_value AS STRING))) IN (
            SELECT code FROM input_condition_codes
          )
          OR REPLACE(UPPER(TRIM(CAST(co.condition_source_value AS STRING))), '.', '') IN (
            SELECT code_nodot FROM input_condition_codes
          )

          OR UPPER(TRIM(CAST(c_standard.concept_code AS STRING))) IN (
            SELECT code FROM input_condition_codes
          )
          OR REPLACE(UPPER(TRIM(CAST(c_standard.concept_code AS STRING))), '.', '') IN (
            SELECT code_nodot FROM input_condition_codes
          )

          OR UPPER(TRIM(CAST(c_source.concept_code AS STRING))) IN (
            SELECT code FROM input_condition_codes
          )
          OR REPLACE(UPPER(TRIM(CAST(c_source.concept_code AS STRING))), '.', '') IN (
            SELECT code_nodot FROM input_condition_codes
          )
        )
      GROUP BY co.person_id
    ),

    last_procedures AS (
      SELECT
        po.person_id,
        MIN(po.procedure_date) AS first_procedure_date,
        MAX(po.procedure_date) AS last_procedure_date
      FROM `{dataset_id}.procedure_occurrence` po
      JOIN input_people ip
        ON po.person_id = ip.person_id
      LEFT JOIN `{dataset_id}.concept` c_standard
        ON po.procedure_concept_id = c_standard.concept_id
      LEFT JOIN `{dataset_id}.concept` c_source
        ON po.procedure_source_concept_id = c_source.concept_id
      WHERE po.procedure_date IS NOT NULL
        AND (
          @match_all_procedures

          OR po.procedure_concept_id IN UNNEST(@procedure_concept_ids)
          OR po.procedure_source_concept_id IN UNNEST(@procedure_concept_ids)

          OR UPPER(TRIM(CAST(po.procedure_source_value AS STRING))) IN (
            SELECT code FROM input_procedure_codes
          )
          OR REPLACE(UPPER(TRIM(CAST(po.procedure_source_value AS STRING))), '.', '') IN (
            SELECT code_nodot FROM input_procedure_codes
          )

          OR UPPER(TRIM(CAST(c_standard.concept_code AS STRING))) IN (
            SELECT code FROM input_procedure_codes
          )
          OR REPLACE(UPPER(TRIM(CAST(c_standard.concept_code AS STRING))), '.', '') IN (
            SELECT code_nodot FROM input_procedure_codes
          )

          OR UPPER(TRIM(CAST(c_source.concept_code AS STRING))) IN (
            SELECT code FROM input_procedure_codes
          )
          OR REPLACE(UPPER(TRIM(CAST(c_source.concept_code AS STRING))), '.', '') IN (
            SELECT code_nodot FROM input_procedure_codes
          )
        )
      GROUP BY po.person_id
    ),

    last_lab_measurements AS (
      SELECT
        m.person_id,
        MIN(m.measurement_date) AS first_lab_measurement_date,
        MAX(m.measurement_date) AS last_lab_measurement_date
      FROM `{dataset_id}.measurement` m
      JOIN input_people ip
        ON m.person_id = ip.person_id
      LEFT JOIN `{dataset_id}.concept` c_standard
        ON m.measurement_concept_id = c_standard.concept_id
      LEFT JOIN `{dataset_id}.concept` c_source
        ON m.measurement_source_concept_id = c_source.concept_id
      WHERE m.measurement_date IS NOT NULL
        AND (
          @match_all_measurements

          OR m.measurement_concept_id IN UNNEST(@measurement_concept_ids)
          OR m.measurement_source_concept_id IN UNNEST(@measurement_concept_ids)

          OR UPPER(TRIM(CAST(m.measurement_source_value AS STRING))) IN (
            SELECT code FROM input_measurement_codes
          )
          OR REPLACE(UPPER(TRIM(CAST(m.measurement_source_value AS STRING))), '.', '') IN (
            SELECT code_nodot FROM input_measurement_codes
          )

          OR UPPER(TRIM(CAST(c_standard.concept_code AS STRING))) IN (
            SELECT code FROM input_measurement_codes
          )
          OR REPLACE(UPPER(TRIM(CAST(c_standard.concept_code AS STRING))), '.', '') IN (
            SELECT code_nodot FROM input_measurement_codes
          )

          OR UPPER(TRIM(CAST(c_source.concept_code AS STRING))) IN (
            SELECT code FROM input_measurement_codes
          )
          OR REPLACE(UPPER(TRIM(CAST(c_source.concept_code AS STRING))), '.', '') IN (
            SELECT code_nodot FROM input_measurement_codes
          )
        )
      GROUP BY m.person_id
    )

    SELECT
      ip.person_id,
      lc.first_condition_date,
      lp.first_procedure_date,
      lm.first_lab_measurement_date,

      CASE
        WHEN lc.first_condition_date IS NULL
         AND lp.first_procedure_date IS NULL
         AND lm.first_lab_measurement_date IS NULL
        THEN NULL
        ELSE LEAST(
          COALESCE(lc.first_condition_date, DATE '9999-12-31'),
          COALESCE(lp.first_procedure_date, DATE '9999-12-31'),
          COALESCE(lm.first_lab_measurement_date, DATE '9999-12-31')
        )
      END AS first_any_code_date,

      lc.last_condition_date,
      lp.last_procedure_date,
      lm.last_lab_measurement_date,

      CASE
        WHEN lc.last_condition_date IS NULL
         AND lp.last_procedure_date IS NULL
         AND lm.last_lab_measurement_date IS NULL
        THEN NULL
        ELSE GREATEST(
          COALESCE(lc.last_condition_date, DATE '0001-01-01'),
          COALESCE(lp.last_procedure_date, DATE '0001-01-01'),
          COALESCE(lm.last_lab_measurement_date, DATE '0001-01-01')
        )
      END AS last_any_code_date

    FROM input_people ip
    LEFT JOIN last_conditions lc
      ON ip.person_id = lc.person_id
    LEFT JOIN last_procedures lp
      ON ip.person_id = lp.person_id
    LEFT JOIN last_lab_measurements lm
      ON ip.person_id = lm.person_id
    ORDER BY ip.person_id
    """

    client = bigquery.Client()

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter(
                "person_ids",
                "INT64",
                cleaned_person_ids,
            ),

            bigquery.ArrayQueryParameter(
                "condition_codes",
                "STRING",
                condition_codes_clean or [],
            ),
            bigquery.ArrayQueryParameter(
                "procedure_codes",
                "STRING",
                procedure_codes_clean or [],
            ),
            bigquery.ArrayQueryParameter(
                "measurement_codes",
                "STRING",
                measurement_codes_clean or [],
            ),

            bigquery.ArrayQueryParameter(
                "condition_concept_ids",
                "INT64",
                condition_ids_clean or [],
            ),
            bigquery.ArrayQueryParameter(
                "procedure_concept_ids",
                "INT64",
                procedure_ids_clean or [],
            ),
            bigquery.ArrayQueryParameter(
                "measurement_concept_ids",
                "INT64",
                measurement_ids_clean or [],
            ),

            bigquery.ScalarQueryParameter(
                "match_all_conditions",
                "BOOL",
                match_all_conditions,
            ),
            bigquery.ScalarQueryParameter(
                "match_all_procedures",
                "BOOL",
                match_all_procedures,
            ),
            bigquery.ScalarQueryParameter(
                "match_all_measurements",
                "BOOL",
                match_all_measurements,
            ),
        ]
    )

    return client.query(query, job_config=job_config).to_dataframe()

In [ ]:
%%time
# Pull EHR date bounds, verify pre-Fitbit history, and save the final ML input

# No code filters are applied, so these are the first and last dates of any record in each domain
cohort_ids = daily_activity_stats_ehr['person_id'].astype(int).unique().tolist()
print(f"Pulling EHR date bounds for {len(cohort_ids)} participants")

ehr_date_bounds_df = pull_ehr_code_date_bounds(cohort_ids)

date_cols = [
    'first_condition_date', 'first_procedure_date', 'first_lab_measurement_date',
    'first_any_code_date', 'last_condition_date', 'last_procedure_date',
    'last_lab_measurement_date', 'last_any_code_date',
]
for col in date_cols:
    ehr_date_bounds_df[col] = pd.to_datetime(ehr_date_bounds_df[col])

# Safe to rerun: drop EHR columns left over from an earlier run of this cell before merging again
old_cols = [c for c in daily_activity_stats_ehr.columns
            if c.startswith(('first_', 'last_', 'has_ehr_', 'days_to_last'))]
daily_activity_stats_ehr = daily_activity_stats_ehr.drop(columns=old_cols)

daily_activity_stats_ehr = daily_activity_stats_ehr.merge(
    ehr_date_bounds_df.astype({'person_id': int}),
    on='person_id',
    how='left'
)

# Check that every participant has an EHR record on or before their first Fitbit day
daily_activity_stats_ehr['first_fitbit_date'] = (
    daily_activity_stats_ehr.groupby('person_id')['date'].transform('min')
)
daily_activity_stats_ehr['has_ehr_before_or_on_first_fitbit'] = (
    daily_activity_stats_ehr['first_any_code_date'].notna()
    & (daily_activity_stats_ehr['first_any_code_date'] <= daily_activity_stats_ehr['first_fitbit_date'])
)
ehr_history_check = (
    daily_activity_stats_ehr[
        ['person_id', 'first_fitbit_date', 'first_any_code_date', 'has_ehr_before_or_on_first_fitbit']
    ]
    .drop_duplicates('person_id')
)
missing_ehr_history = ehr_history_check[
    ~ehr_history_check['has_ehr_before_or_on_first_fitbit']
]
print(
    f"Participants with EHR history on/before first Fitbit date: "
    f"{len(ehr_history_check) - len(missing_ehr_history)}/{len(ehr_history_check)}"
)
# Participants without EHR history before their first Fitbit day are kept and flagged;
# filter them with has_ehr_before_or_on_first_fitbit in the downstream notebooks
if not missing_ehr_history.empty:
    missing_ehr_history.to_csv('missing_ehr_history_003.csv', index=False)
    print(f"WARNING: {len(missing_ehr_history)} participants have no EHR record on or before their first Fitbit date. "
          "Kept in the table; filter with has_ehr_before_or_on_first_fitbit. List saved to missing_ehr_history_003.csv")

# Days from each Fitbit day to the participant's last EHR record
# (negative means the Fitbit day is after the last record)
daily_activity_stats_ehr['days_to_last_encounter'] = (
    daily_activity_stats_ehr['last_any_code_date'] - daily_activity_stats_ehr['date']
).dt.days

print("\nCoverage (fraction of rows with a date):")
print(daily_activity_stats_ehr[date_cols].notna().mean())
print("\ndays_to_last_encounter:")
print(daily_activity_stats_ehr['days_to_last_encounter'].describe())

# Final save. The model notebooks read this file; the earlier to_parquet is an intermediate checkpoint.
daily_activity_stats_ehr.to_parquet(f'{bucket_}/{file_path_to_save}.pq')
print(f"\nSaved {len(daily_activity_stats_ehr)} rows x {daily_activity_stats_ehr.shape[1]} cols to {bucket_}/{file_path_to_save}.pq")

## Section 1.4: Validation

This section does not modify the data. It reviews the expected columns, summary statistics, and feature distributions, and checks that the in-scope sleep-level minutes add up to `sleep_duration`, that no minutes fall on unmapped sleep codes, and that the awake HR proportions add up to 1.

In [ ]:
# Check that the expected columns exist and print summary statistics
latent_cols = {
    'summary.step_count_nw': 'latent_0 (steps_count)',
    'summary.mean_sleep_hr': 'latent_1 (sleep_hr_mean_all)',
    'summary.std_sleep_hr': 'latent_2 (sleep_hr_std)',
    'summary.mean_non_sleep_hr': 'latent_3 (wake_hr_mean_all)',
    'summary.std_non_sleep_hr': 'latent_4 (wake_hr_std)',
    'summary.awake_minutes': 'latent_5 (awake = short awakenings, wake+awake labels)',
    'summary.asleep_grpd_rstls': 'latent_6 (asleep_grpd_rstls)',
    'summary.age': 'latent_7 (age)',
    'bmi': 'latent_8 (bmi)',
    'is_male': 'latent_9 (is_male)',
    'summary.sleep_start_time': 'sleep_start_time',
    'summary.pct_awake_hr_low': 'pct_awake_hr_low',
    'summary.pct_awake_hr_med': 'pct_awake_hr_med',
    'summary.pct_awake_hr_high': 'pct_awake_hr_high',
    'summary.sleep_duration': 'sleep_duration (minutes in a sleep record)',
}

sleep_level_cols = {}
for lvl in SLEEP_LEVELS:
    sleep_level_cols[f'summary.{lvl}'] = f'{lvl} minutes (main sleep)'
    sleep_level_cols[f'summary.nonmain_{lvl}'] = f'{lvl} minutes (non-main sleep)'
    sleep_level_cols[f'summary.{lvl}_total'] = f'{lvl} minutes (follows USE_MAIN_SLEEP_ONLY)'
sleep_level_cols['date_info.day_of_week'] = 'weekday name'

ehr_date_cols = [
    'first_fitbit_date',
    'first_condition_date',
    'first_procedure_date',
    'first_lab_measurement_date',
    'first_any_code_date',
    'last_condition_date',
    'last_procedure_date',
    'last_lab_measurement_date',
    'last_any_code_date',
]
ehr_cols = {
    'first_fitbit_date': 'first Fitbit date',
    'first_condition_date': 'first condition date',
    'first_procedure_date': 'first procedure date',
    'first_lab_measurement_date': 'first lab/measurement date',
    'first_any_code_date': 'first EHR code date (any domain)',
    'last_condition_date': 'last condition date',
    'last_procedure_date': 'last procedure date',
    'last_lab_measurement_date': 'last lab/measurement date',
    'last_any_code_date': 'last encounter (any domain)',
    'days_to_last_encounter': 'days from this day to last encounter',
    'has_ehr_before_or_on_first_fitbit': 'EHR history check passed',
}

all_cols = {**latent_cols, **sleep_level_cols, **ehr_cols}

print("=== Column Check ===")
for col, label in all_cols.items():
    status = "FOUND" if col in daily_activity_stats_ehr.columns else "MISSING"
    print(f"  {status}: {col}  ->  {label}")

found = [c for c in all_cols if c in daily_activity_stats_ehr.columns and c not in ehr_date_cols]
print(f"\n=== Stats for {len(found)}/{len(all_cols)} columns ===")
daily_activity_stats_ehr[found].describe()

In [ ]:
%%time
# Plot feature distributions and run two consistency checks: sleep levels sum to sleep_duration, HR buckets sum to 1
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (16, 16)
fig, axes = plt.subplots(4, 4, figsize=(20, 18))
fig.suptitle(f'Latent Feature Validation - {daily_activity_stats_ehr["person_id"].nunique()} People', fontsize=16, fontweight='bold')
plots = [
    ('summary.step_count_nw', 'Steps (non-wear adjusted)', 'Expected: 5K-10K avg'),
    ('summary.mean_sleep_hr', 'Mean Sleep HR (bpm)', 'Expected: 55-70 bpm'),
    ('summary.std_sleep_hr', 'Std Sleep HR (bpm)', 'Expected: 3-6 bpm'),
    ('summary.mean_non_sleep_hr', 'Mean Wake HR (bpm)', 'Expected: 70-85 bpm'),
    ('summary.std_non_sleep_hr', 'Std Wake HR (bpm)', 'Expected: 8-15 bpm'),
    ('summary.awake_minutes', 'Short Awakenings (wake+awake)', 'Expected: ~10-60 min'),
    ('summary.asleep_grpd_rstls', 'Asleep Minutes', 'Expected: ~420-480 (7-8hrs)'),
    ('summary.sleep_start_time', 'Sleep Start (minute of day)', 'Expected: ~1300-1400 (9-11pm)'),
    ('summary.pct_awake_hr_low', '% Awake Low HR', 'Expected: ~70-80%'),
    ('summary.pct_awake_hr_med', '% Awake Med HR', 'Expected: ~15-20%'),
    ('summary.pct_awake_hr_high', '% Awake High HR', 'Expected: ~3-8%'),
    ('summary.age', 'Age', 'Expected: 18-90'),
    ('bmi', 'BMI', 'Expected: 18-40 avg'),
    ('is_male', 'Is Male', 'Expected: 0 or 1'),
]
for idx, (col, title, note) in enumerate(plots):
    ax = axes[idx // 4][idx % 4]
    data = daily_activity_stats_ehr[col].dropna()
    ax.hist(data, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(f'{title}\n({note})', fontsize=10)
    ax.axvline(data.mean(), color='red', linestyle='--', label=f'mean={data.mean():.1f}')
    ax.axvline(data.median(), color='orange', linestyle='--', label=f'median={data.median():.1f}')
    ax.legend(fontsize=8)

# In-scope sleep levels should add up to sleep_duration
scope_cols = [f'summary.{lvl}' for lvl in SLEEP_LEVELS]
if not USE_MAIN_SLEEP_ONLY:
    scope_cols += [f'summary.nonmain_{lvl}' for lvl in SLEEP_LEVELS]
level_sum_diff = daily_activity_stats_ehr[scope_cols].sum(axis=1) - daily_activity_stats_ehr['summary.sleep_duration']

ax = axes[3][2]
ax.hist(level_sum_diff.dropna(), bins=50, color='green', edgecolor='white', alpha=0.8)
ax.set_title('sum(sleep levels) - sleep_duration\n(Should be exactly 0)', fontsize=10)
ax.axvline(0, color='red', linestyle='--', label='0')
ax.legend(fontsize=8)
axes[3][3].axis('off')
plt.tight_layout()
plt.savefig('latent_validation.png', dpi=150, bbox_inches='tight')
plt.show()

# Mean minutes per sleep level, main vs non-main
main_means = [daily_activity_stats_ehr[f'summary.{lvl}'].mean() for lvl in SLEEP_LEVELS]
nonmain_means = [daily_activity_stats_ehr[f'summary.nonmain_{lvl}'].mean() for lvl in SLEEP_LEVELS]
x = np.arange(len(SLEEP_LEVELS))
fig2, ax2 = plt.subplots(figsize=(10, 5))
ax2.bar(x - 0.2, main_means, width=0.4, label='main sleep', color='steelblue')
ax2.bar(x + 0.2, nonmain_means, width=0.4, label='non-main sleep', color='indianred')
ax2.set_xticks(x)
ax2.set_xticklabels(SLEEP_LEVELS)
ax2.set_ylabel('mean minutes per day')
ax2.set_title('Minutes per sleep level')
ax2.legend()
plt.tight_layout()
plt.savefig('sleep_level_validation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== Quick Validation Summary ===")
print(f"Total rows: {len(daily_activity_stats_ehr)}")
print(f"Unique people: {daily_activity_stats_ehr['person_id'].nunique()}")
for col, title, note in plots:
    data = daily_activity_stats_ehr[col]
    print(f"\n{title}:")
    print(f"  NaN rate: {data.isna().mean()*100:.1f}%  |  mean: {data.mean():.2f}  |  median: {data.median():.2f}")
for lvl in SLEEP_LEVELS:
    main_minutes = daily_activity_stats_ehr[f'summary.{lvl}']
    nonmain_minutes = daily_activity_stats_ehr[f'summary.nonmain_{lvl}']
    print(f"\n{lvl}: main mean {main_minutes.mean():.1f} min  |  non-main mean {nonmain_minutes.mean():.1f} min")

print(f"\n=== Sleep Level Sum Check (USE_MAIN_SLEEP_ONLY={USE_MAIN_SLEEP_ONLY}) ===")
print(f"  Mean difference vs sleep_duration: {level_sum_diff.mean():.6f} (should be 0.0)")
print(f"  All exactly 0: {(level_sum_diff.dropna() == 0).all()}")
print(f"  Minutes on unmapped sleep codes: {daily_activity_stats_ehr['summary.unmapped_sleep_minutes'].sum():.0f} (should be 0)")

hr_sum = (daily_activity_stats_ehr['summary.pct_awake_hr_low'] +
          daily_activity_stats_ehr['summary.pct_awake_hr_med'] +
          daily_activity_stats_ehr['summary.pct_awake_hr_high'])
print(f"\n=== HR Bucket Sum Check ===")
print(f"  Mean sum: {hr_sum.mean():.6f} (should be 1.0)")
print(f"  All equal 1.0: {np.isclose(hr_sum.dropna(), 1.0).all()}")

## Section 1.5: Next Steps

Use the saved table as input to the transformer, Cox regression, and logistic regression model-training notebooks.